**Ingest Sprints.csv file**
1. Read file using spark dataframe reader API
2.add Metadata Columns 
-   source file
- ingestion Timestamp
3. write to bonze delta table

Note: JSON is in multi line format

In [0]:
%run ../00.Common/01.environment-config

In [0]:
%run ../00.Common/02.bronze-helpers


In [0]:
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_nameone}.{bronze_schema}.sprints"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, FloatType


sprints_schema = StructType([
    StructField("date", DateType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", StringType()),
    StructField("url", StringType()),
    StructField("constructorId",StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType()),

])


sprints_df = (
    spark.read.format("json")
    .schema(sprints_schema)
    .option("mode", "FAILFAST")
    .option("multiLine", "true")
    .load(source_file)
)

sprints_final_df = add_ingestion_metadata(sprints_df)
display(sprints_final_df)

In [0]:
sprints_final_df.write.format('delta').mode('overwrite').saveAsTable(table_name)